# 01 — Data Cleaning & Preprocessing

**Project:** Uber Data India (2024) Analysis

This notebook loads the raw Uber dataset, inspects it, cleans it, engineers features, and saves the processed data.

---

In [1]:
# ============================================================
# CELL 1: Import Libraries & Setup Paths
# ============================================================
import sys
import os

# Add project root to path so we can import from src/
notebook_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in dir() else '.'))
project_root = os.path.dirname(notebook_dir)
sys.path.insert(0, project_root)

from src.data_loader import load_uber_data, inspect_data, save_processed_data
from src.preprocessing import clean_dataset
from src.feature_engineering import engineer_all_features

import pandas as pd
import numpy as np

print('✅ All modules imported successfully!')

✅ All modules imported successfully!


## Step 0: Check Your Column Names (IMPORTANT)

Run this first to see what columns your actual CSV file has. If names differ from our expected names, the code will auto-detect and map them.

In [2]:
# ============================================================
# CELL 2: Check Column Names
# ============================================================
# Try to load and see what columns you have
import pandas as pd
try:
    df_check = pd.read_csv('../data/raw/uber_data_india_2024.csv', nrows=5)
    print('📋 ACTUAL COLUMN NAMES IN YOUR CSV:')
    print('=' * 50)
    for i, col in enumerate(df_check.columns, 1):
        print(f'  {i:2}. {repr(col)}')
    print(f"\n💡 If these don't match expected names, the code will auto-map them.")
except FileNotFoundError:
    print("❌ File not found at ../data/raw/uber_data_india_2024.csv")
    print("   Please check your file path and update it below.")

📋 ACTUAL COLUMN NAMES IN YOUR CSV:
   1. 'Date'
   2. 'Time'
   3. 'Booking ID'
   4. 'Booking Status'
   5. 'Customer ID'
   6. 'Vehicle Type'
   7. 'Pickup Location'
   8. 'Drop Location'
   9. 'Avg VTAT'
  10. 'Avg CTAT'
  11. 'Cancelled Rides by Customer'
  12. 'Reason for cancelling by Customer'
  13. 'Cancelled Rides by Driver'
  14. 'Driver Cancellation Reason'
  15. 'Incomplete Rides'
  16. 'Incomplete Rides Reason'
  17. 'Booking Value'
  18. 'Ride Distance'
  19. 'Driver Ratings'
  20. 'Customer Rating'
  21. 'Payment Method'

💡 If these don't match expected names, the code will auto-map them.


## Step 1: Load the Dataset

Load the 148,770 Uber ride bookings from `data/raw/`. If your file is elsewhere, update the path.

In [3]:
# ============================================================
# CELL 3: Load Raw Data
# ============================================================
# Update this path if your file is in a different location
DATA_PATH = '../data/raw/uber_data_india_2024.csv'

df = load_uber_data(DATA_PATH)

# Quick peek
df.head()

✅ Dataset loaded successfully from: ../data/raw/uber_data_india_2024.csv
   Shape: 150,000 rows × 21 columns


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


## Step 2: Initial Inspection

Check shape, dtypes, missing values, duplicates, and basic statistics.

In [4]:
# ============================================================
# CELL 4: Inspect Data
# ============================================================
inspect_data(df)


📊 DATASET OVERVIEW

Shape: 150,000 rows × 21 columns

📋 Column Names & Data Types:
Date                                     str
Time                                     str
Booking ID                               str
Booking Status                           str
Customer ID                              str
Vehicle Type                             str
Pickup Location                          str
Drop Location                            str
Avg VTAT                             float64
Avg CTAT                             float64
Cancelled Rides by Customer          float64
Reason for cancelling by Customer        str
Cancelled Rides by Driver            float64
Driver Cancellation Reason               str
Incomplete Rides                     float64
Incomplete Rides Reason                  str
Booking Value                        float64
Ride Distance                        float64
Driver Ratings                       float64
Customer Rating                      float64
Payment Method  

## Step 3: Clean the Dataset

Run the full cleaning pipeline:
- Auto-detect and rename columns to standard names
- Remove duplicates
- Fix data types (datetime, numeric, boolean)
- Handle missing values
- Cap outliers (IQR method)
- Standardise text columns

In [5]:
# ============================================================
# CELL 5: Clean Data
# ============================================================
df_clean = clean_dataset(df)

print('\n📊 Cleaned Dataset Shape:', df_clean.shape)
df_clean.head()


🧹 DATA CLEANING PIPELINE
📋 COLUMN NAME DETECTION

✅ Found 21 matching columns:
   'Date'
   'Time'
   'Booking ID'
   'Booking Status'
   'Customer ID'
   'Vehicle Type'
   'Pickup Location'
   'Drop Location'
   'Avg VTAT'
   'Avg CTAT'
   'Cancelled Rides by Customer' → 'Customer Cancellation'
   'Reason for cancelling by Customer' → 'Customer Cancellation Reason'
   'Cancelled Rides by Driver' → 'Driver Cancellation'
   'Driver Cancellation Reason'
   'Incomplete Rides' → 'Incomplete Ride'
   'Incomplete Rides Reason' → 'Incomplete Ride Reason'
   'Booking Value'
   'Ride Distance'
   'Driver Ratings' → 'Driver Rating'
   'Customer Rating'
   'Payment Method'

🔧 Renamed 21 columns to standardized names.
🗑️  Removed 0 duplicate rows

🔧 Fixing Data Types...
   Created 'Booking_DateTime' from Date + Time

🔧 Handling Missing Values...
   Avg VTAT: Imputed 10500 missing values with median (8.30)
   Avg CTAT: Imputed 48000 missing values with median (28.80)
   Booking Value: Imputed 4800

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Driver Cancellation,Driver Cancellation Reason,Incomplete Ride,Incomplete Ride Reason,Booking Value,Ride Distance,Driver Rating,Customer Rating,Payment Method,Booking_DateTime
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",Ebike,Palam Vihar,Jhilmil,8.3,28.8,...,False,Not Cancelled,False,NaN,414.0,23.72,4.3,4.5,Upi,2024-03-23 12:29:38
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,False,Not Cancelled,True,Vehicle Breakdown,237.0,5.73,4.3,4.5,Upi,2024-11-29 18:01:39
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,False,Not Cancelled,False,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,False,Not Cancelled,False,NaN,416.0,34.02,4.6,5.0,Upi,2024-10-21 17:17:25
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,False,Not Cancelled,False,NaN,737.0,48.21,4.1,4.3,Upi,2024-09-16 22:08:00


## Step 4: Feature Engineering

Create new features for analysis. The code will skip any features whose required columns are missing.

In [6]:
# ============================================================
# CELL 6: Engineer Features
# ============================================================
df_engineered = engineer_all_features(df_clean)

print('\n📊 Engineered Dataset Shape:', df_engineered.shape)
print('\n📋 All Columns:')
for i, col in enumerate(df_engineered.columns, 1):
    print(f'  {i:2}. {col}')

df_engineered.head()


⚙️  FEATURE ENGINEERING

🔧 Extracting Temporal Features...
   Created: Booking_Hour, Booking_Day, Booking_Month, Booking_Weekday, Is_Weekend, Time_of_Day

🔧 Creating Cancellation Features...
   Created: Is_Cancelled (37,500 cancellations found)
   Created: Cancellation_Type

🔧 Creating Revenue Features...
   Created: Revenue_per_km, High_Value_Ride (threshold: ₹521.00)

🔧 Creating Operational Features...
   Created: Wait_Time_Category
   Created: Distance_Category

🔧 Creating Rating Features...
   Created: Avg_Rating, Rating_Category

✅ Feature engineering complete!
   Final shape: 150,000 rows × 36 columns

📊 Engineered Dataset Shape: (150000, 36)

📋 All Columns:
   1. Date
   2. Time
   3. Booking ID
   4. Booking Status
   5. Customer ID
   6. Vehicle Type
   7. Pickup Location
   8. Drop Location
   9. Avg VTAT
  10. Avg CTAT
  11. Customer Cancellation
  12. Customer Cancellation Reason
  13. Driver Cancellation
  14. Driver Cancellation Reason
  15. Incomplete Ride
  16. Incompl

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Is_Weekend,Time_of_Day,Is_Cancelled,Cancellation_Type,Revenue_per_km,High_Value_Ride,Wait_Time_Category,Distance_Category,Avg_Rating,Rating_Category
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",Ebike,Palam Vihar,Jhilmil,8.3,28.8,...,True,Afternoon,False,None,17.453626,False,Moderate (5-10 min),Long (10-25 km),4.4,Good
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,False,Evening,False,None,41.361257,False,Quick (<5 min),Medium (3-10 km),4.4,Good
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,False,Morning,False,None,46.170839,True,Long (10-20 min),Long (10-25 km),4.9,Excellent
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,False,Evening,False,None,12.228101,False,Long (10-20 min),Very Long (>25 km),4.8,Excellent
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,False,Night,False,None,15.287285,True,Moderate (5-10 min),Very Long (>25 km),4.2,Good


## Step 5: Save Processed Data

Save the cleaned and engineered dataset to `data/processed/` for use in subsequent notebooks.

In [7]:
# ============================================================
# CELL 7: Save Cleaned Data
# ============================================================
save_processed_data(df_engineered, '../data/processed/engineered_uber_data.csv')

# Also save intermediate cleaned version
save_processed_data(df_clean, '../data/processed/cleaned_uber_data.csv')


💾 Processed data saved to: ../data/processed/engineered_uber_data.csv
   Shape: 150,000 rows × 36 columns

💾 Processed data saved to: ../data/processed/cleaned_uber_data.csv
   Shape: 150,000 rows × 36 columns


## Step 6: Quick Validation — Cancellation Overview

A quick sanity check to confirm our cancellation features work correctly.

In [8]:
# ============================================================
# CELL 8: Cancellation Summary
# ============================================================
if 'Is_Cancelled' in df_engineered.columns:
    print('📊 CANCELLATION OVERVIEW')
    print('=' * 50)
    
    total = len(df_engineered)
    cancelled = df_engineered['Is_Cancelled'].sum()
    print(f'Total Bookings:      {total:,}')
    print(f'Cancelled Rides:     {cancelled:,} ({cancelled/total*100:.2f}%)')
    print(f'Completed Rides:     {total-cancelled:,} ({(total-cancelled)/total*100:.2f}%)')
    
    if 'Cancellation_Type' in df_engineered.columns:
        print('\n📋 Cancellation Type Breakdown:')
        print(df_engineered['Cancellation_Type'].value_counts())
    
    if 'Vehicle Type' in df_engineered.columns:
        print('\n🚗 Cancellation Rate by Vehicle Type:')
        cancel_rate = df_engineered.groupby('Vehicle Type')['Is_Cancelled'].agg(['count', 'sum', 'mean'])
        cancel_rate.columns = ['Total_Bookings', 'Cancelled', 'Cancellation_Rate']
        cancel_rate['Cancellation_Rate'] = cancel_rate['Cancellation_Rate'] * 100
        print(cancel_rate.sort_values('Cancellation_Rate', ascending=False))
else:
    print('⚠️ Is_Cancelled column not found. Cancellation features may not have been created.')

📊 CANCELLATION OVERVIEW
Total Bookings:      150,000
Cancelled Rides:     37,500 (25.00%)
Completed Rides:     112,500 (75.00%)

📋 Cancellation Type Breakdown:
Cancellation_Type
None        112500
Driver       27000
Customer     10500
Name: count, dtype: int64

🚗 Cancellation Rate by Vehicle Type:
               Total_Bookings  Cancelled  Cancellation_Rate
Vehicle Type                                               
Go Sedan                27141       6863          25.286467
Bike                    22517       5652          25.101035
Premier Sedan           18111       4516          24.935122
Go Mini                 29806       7427          24.917802
Auto                    37419       9323          24.915150
Ebike                   10557       2630          24.912380
Uber Xl                  4449       1089          24.477411


## Step 7: Quick Validation — Revenue Overview

Check revenue distribution and top-performing vehicle types.

In [9]:
# ============================================================
# CELL 9: Revenue Summary
# ============================================================
if 'Booking Value' in df_engineered.columns:
    print('💰 REVENUE OVERVIEW')
    print('=' * 50)
    
    print(f'Average Booking Value: ₹{df_engineered["Booking Value"].mean():.2f}')
    print(f'Median Booking Value:  ₹{df_engineered["Booking Value"].median():.2f}')
    print(f'Total Revenue:         ₹{df_engineered["Booking Value"].sum():,.2f}')
    
    if 'Vehicle Type' in df_engineered.columns:
        print('\n📋 Revenue by Vehicle Type:')
        revenue = df_engineered.groupby('Vehicle Type')['Booking Value'].agg(['sum', 'mean', 'count'])
        revenue.columns = ['Total_Revenue', 'Avg_Fare', 'Total_Rides']
        revenue = revenue.sort_values('Total_Revenue', ascending=False)
        print(revenue)
else:
    print('⚠️ Booking Value column not found.')

💰 REVENUE OVERVIEW
Average Booking Value: ₹438.25
Median Booking Value:  ₹414.00
Total Revenue:         ₹65,737,174.50

📋 Revenue by Vehicle Type:
               Total_Revenue    Avg_Fare  Total_Rides
Vehicle Type                                         
Auto            1.639137e+07  438.049427        37419
Go Mini         1.305624e+07  438.040625        29806
Go Sedan        1.192992e+07  439.553535        27141
Bike            9.870926e+06  438.376604        22517
Premier Sedan   7.961029e+06  439.568694        18111
Ebike           4.589993e+06  434.781910        10557
Uber Xl         1.937694e+06  435.534811         4449


---

## ✅ Notebook Complete!

**Outputs generated:**
- `data/processed/cleaned_uber_data.csv` — cleaned dataset
- `data/processed/engineered_uber_data.csv` — cleaned + engineered features

**Next:** Open `02_eda_booking_trends.ipynb` for full exploratory data analysis and visualisations.